In [ ]:
!nvidia-smi

# Install required packages
!pip install transformers torch tqdm sentencepiece safetensors -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

Mon Mar  2 13:22:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
DATA_DIR = '/content/'
OUTPUT_DIR = '/content/embeddings'

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
import logging
from transformers import AutoTokenizer, AutoModel, AutoConfig
import torch

In [ ]:
# Setup logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Device setup-
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Model configuration
MODEL_NAME = "SPhilBERTa-Intertext"
MODEL_ID = "julian-schelb/SPhilBerta-emb-lat-intertext-v1"
BATCH_SIZE = 32  # Adjust based on GPU memory
MAX_LENGTH = 512  # Maximum sequence length
USE_MEAN_POOLING = True  # Token averaging for LaBERTa

print(f"Model: {MODEL_NAME}")
print(f"Model ID: {MODEL_ID}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max length: {MAX_LENGTH}")

Model: SPhilBERTa-Intertext
Model ID: julian-schelb/SPhilBerta-emb-lat-intertext-v1
Batch size: 32
Max length: 512


In [ ]:
def load_chunks_json(filepath: str) -> Tuple[List[Dict], Dict]:
    """
    Load chunks from JSON file.

    Args:
        filepath: Path to the .json file (with or without extension)

    Returns:
        Tuple of (chunks list, metadata dict)
    """
    # Ensure .json extension
    if not filepath.endswith('.json'):
        filepath = f"{filepath}.json"

    print(f"Loading {Path(filepath).name}...")

    # Load JSON file
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Extract chunks and metadata
    chunks = data.get('chunks', [])
    metadata = data.get('metadata', {})

    print(f"✓ Loaded {len(chunks):,} chunks")

    # Show retention rate if available in metadata
    if metadata.get('filtering', {}).get('retention_rate'):
        retention = metadata['filtering']['retention_rate'] * 100
        print(f"  Retention rate: {retention:.1f}%")

    print()

    return chunks, metadata


# Load BDC and PSC chunks
bdc_chunks, bdc_metadata = load_chunks_json(f"{DATA_DIR}/VD_bdc_chunks.json")
psc_chunks, psc_metadata = load_chunks_json(f"{DATA_DIR}/VD_psc_chunks.json")

print(f"Total chunks to embed:")
print(f"  BDC (queries): {len(bdc_chunks):,}")
print(f"  PSC (candidates): {len(psc_chunks):,}")

Loading VD_bdc_chunks.json...
✓ Loaded 19,466 chunks

Loading VD_psc_chunks.json...
✓ Loaded 46,408 chunks

Total chunks to embed:
  BDC (queries): 19,466
  PSC (candidates): 46,408


In [ ]:
from transformers import AutoTokenizer, RobertaModel
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch
import torch.nn as nn

print(f"Loading {MODEL_NAME}...")
print(f"Model ID: {MODEL_ID}\n")

# Load tokenizer
BASE_ENCODER = "julian-schelb/SPhilBerta-emb-lat-intertext-v1"
tokenizer = AutoTokenizer.from_pretrained(BASE_ENCODER)
print(f"✓ Tokenizer loaded from {BASE_ENCODER}")

# Step 2: Define RetrieverModel class (matches the architecture)
class RetrieverModel(nn.Module):
    """
    Custom RetrieverModel that wraps RoBERTa with mean pooling.
    Matches the architecture from itserr/LaBERTa-W_VULG-S_VL
    """
    def __init__(self, base_model_name: str, pooling_strategy: str = "mean"):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(base_model_name)
        self.pooling_strategy = pooling_strategy

    def mean_pooling(self, token_embeddings, attention_mask):
        """Mean pooling over token embeddings."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask, **kwargs):
        """Forward pass with mean pooling."""
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)

        if self.pooling_strategy == "mean":
            embeddings = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        else:
            # Fallback to [CLS] token
            embeddings = outputs.last_hidden_state[:, 0, :]

        return embeddings

# Step 3: Initialize model with base encoder
print("Initializing RetrieverModel...")
model = RetrieverModel(BASE_ENCODER, pooling_strategy="mean").to(device)
print(f"✓ Model structure initialized")

# Step 4: Load fine-tuned weights
print("\nLoading fine-tuned weights...")
model_file = hf_hub_download(
    repo_id=MODEL_ID,
    filename="model.safetensors"
)

state_dict = load_file(model_file)
print(f"Checkpoint contains {len(state_dict)} keys")

# Load weights into the model
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)

print(f"✓ Fine-tuned weights loaded")
print(f"  Missing keys: {len(missing_keys)}")
print(f"  Unexpected keys: {len(unexpected_keys)}")

model.eval()

print(f"  Base encoder: {BASE_ENCODER}")
print(f"  Pooling: mean (token averaging)")
print(f"  Device: {next(model.parameters()).device}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading LaBERTa-Vulgate...
Model ID: itserr/LaBERTa-W_VULG-S_VL



config.json:   0%|          | 0.00/674 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: bowphs/LaBerta
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model structure initialized

Loading fine-tuned weights...


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Checkpoint contains 200 keys
✓ Fine-tuned weights loaded
  Missing keys: 199
  Unexpected keys: 200
  Base encoder: bowphs/LaBerta
  Pooling: mean (token averaging)
  Device: cuda:0
  Parameters: 125,978,112


In [ ]:
# ============================================================================
# ENCODING LABERTA
# ============================================================================

def encode_batch(texts: List[str]) -> np.ndarray:
    """
    Encode batch using RetrieverModel (already does mean pooling).
    """
    # Tokenize
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(device)

    # Generate embeddings (model handles mean pooling internally)
    with torch.no_grad():
        embeddings = model(
            input_ids=encoded["input_ids"],
            attention_mask=encoded["attention_mask"]
        )

    # Normalize for cosine similarity
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    return embeddings.cpu().numpy()


def encode_corpus(chunks: List[Dict], corpus_name: str, checkpoint_every: int = 1000) -> np.ndarray:
    """Encode entire corpus in batches."""
    print(f"\n{'='*60}")
    print(f"Encoding {corpus_name}")
    print(f"{'='*60}")
    print(f"Total chunks: {len(chunks):,}")
    print(f"Batch size: {BATCH_SIZE}\n")

    texts = [chunk["text"] for chunk in chunks]
    all_embeddings = []

    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc=f"Encoding {corpus_name}"):
        batch_texts = texts[i:i + BATCH_SIZE]
        batch_embeddings = encode_batch(batch_texts)
        all_embeddings.append(batch_embeddings)

        # Checkpoint
        batch_num = i // BATCH_SIZE
        if batch_num > 0 and batch_num % checkpoint_every == 0:
            checkpoint = np.vstack(all_embeddings)
            np.save(f"{OUTPUT_DIR}/{corpus_name.lower()}_checkpoint.npy", checkpoint)
            print(f"\n✓ Checkpoint: {checkpoint.shape[0]} embeddings")

    embeddings = np.vstack(all_embeddings)

    print(f"\n✓ Encoding complete")
    print(f"  Shape: {embeddings.shape}")
    print(f"  Dimension: {embeddings.shape[1]}")
    print(f"  Memory: {embeddings.nbytes / 1024**2:.2f} MB")

    return embeddings

In [ ]:
bdc_embeddings = encode_corpus(bdc_chunks, "BDC")

# Save BDC embeddings
bdc_output_path = f"{OUTPUT_DIR}/bdc_embeddings_laberta_vulg.npy"
np.save(bdc_output_path, bdc_embeddings)
print(f"\n✓ Saved BDC embeddings to: {bdc_output_path}")

# Save BDC chunk IDs (for mapping embeddings back to chunks)
bdc_ids = [chunk["chunk_id"] for chunk in bdc_chunks]
bdc_ids_path = f"{OUTPUT_DIR}/bdc_ids_laberta_vulg.json"
with open(bdc_ids_path, 'w') as f:
    json.dump(bdc_ids, f, indent=2)
print(f"✓ Saved BDC IDs to: {bdc_ids_path}")


Encoding BDC
Total chunks: 19,466
Batch size: 32



Encoding BDC:   0%|          | 0/609 [00:00<?, ?it/s]


✓ Encoding complete
  Shape: (19466, 768)
  Dimension: 768
  Memory: 57.03 MB

✓ Saved BDC embeddings to: /content/embeddings/bdc_embeddings_laberta_vulg.npy
✓ Saved BDC IDs to: /content/embeddings/bdc_ids_laberta_vulg.json


In [ ]:
psc_embeddings = encode_corpus(psc_chunks, "PSC")

# Save PSC embeddings
psc_output_path = f"{OUTPUT_DIR}/psc_embeddings_laberta_vulg.npy"
np.save(psc_output_path, psc_embeddings)
print(f"\n✓ Saved PSC embeddings to: {psc_output_path}")

# Save PSC chunk IDs
psc_ids = [chunk["chunk_id"] for chunk in psc_chunks]
psc_ids_path = f"{OUTPUT_DIR}/psc_ids_laberta_vulg.json"
with open(psc_ids_path, 'w') as f:
    json.dump(psc_ids, f, indent=2)
print(f"✓ Saved PSC IDs to: {psc_ids_path}")


Encoding PSC
Total chunks: 46,408
Batch size: 32



Encoding PSC:   0%|          | 0/1451 [00:00<?, ?it/s]


✓ Checkpoint: 32032 embeddings

✓ Encoding complete
  Shape: (46408, 768)
  Dimension: 768
  Memory: 135.96 MB

✓ Saved PSC embeddings to: /content/embeddings/psc_embeddings_laberta_vulg.npy
✓ Saved PSC IDs to: /content/embeddings/psc_ids_laberta_vulg.json


In [ ]:
metadata = {
    "model_name": MODEL_NAME,
    "model_id": MODEL_ID,
    "batch_size": BATCH_SIZE,
    "max_length": MAX_LENGTH,
    "device": str(device),
    "bdc": {
        "num_chunks": len(bdc_chunks),
        "embedding_shape": list(bdc_embeddings.shape),
        "embedding_dim": bdc_embeddings.shape[1],
        "file": "bdc_embeddings_laberta_vulg.npy"
    },
    "psc": {
        "num_chunks": len(psc_chunks),
        "embedding_shape": list(psc_embeddings.shape),
        "embedding_dim": psc_embeddings.shape[1],
        "file": "psc_embeddings_laberta_vulg.npy"
    }
}

metadata_path = f"{OUTPUT_DIR}/metadata_laberta_vulg.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Saved metadata to: {metadata_path}")

✓ Saved metadata to: /content/embeddings/metadata_laberta_vulg.json


In [ ]:
#  sanity check
print(f"  BDC embeddings shape matches chunks: {bdc_embeddings.shape[0] == len(bdc_chunks)}")
print(f"  PSC embeddings shape matches chunks: {psc_embeddings.shape[0] == len(psc_chunks)}")
print(f"  BDC IDs count matches: {len(bdc_ids) == len(bdc_chunks)}")
print(f"  PSC IDs count matches: {len(psc_ids) == len(psc_chunks)}")
print(f"  Embedding dimensions match: {bdc_embeddings.shape[1] == psc_embeddings.shape[1]}")

  BDC embeddings shape matches chunks: True
  PSC embeddings shape matches chunks: True
  BDC IDs count matches: True
  PSC IDs count matches: True
  Embedding dimensions match: True


In [ ]:
# ============================================================================
# DIAGNOSTIC: Check embedding files
# ============================================================================

import numpy as np
import os

MODEL_NAME = "laberta_vulg"

def check_embedding_file(filepath):
    """Check if embedding file is complete."""
    print(f"\nChecking: {filepath}")

    # File size
    file_size_mb = os.path.getsize(filepath) / (1024**2)
    print(f"  File size: {file_size_mb:.2f} MB")

    # Try to load with allow_pickle=False
    try:
        arr = np.load(filepath, allow_pickle=False)
        print(f"  ✓ Loaded successfully")
        print(f"  Shape: {arr.shape}")
        print(f"  Total elements: {arr.size:,}")
        return True
    except ValueError as e:
        print(f"  ✗ Load failed: {e}")

        # Check raw data size
        with open(filepath, 'rb') as f:
            # Skip header (128 bytes typically)
            f.seek(128)
            data = f.read()
            num_floats = len(data) // 4  # 4 bytes per float32
            print(f"  Raw data contains ~{num_floats:,} float32 values")

        return False

# Check both files
bdc_path = f"{OUTPUT_DIR}/bdc_embeddings_{MODEL_NAME}.npy"
psc_path = f"{OUTPUT_DIR}/psc_embeddings_{MODEL_NAME}.npy"

print("="*60)
print("EMBEDDING FILE DIAGNOSTICS")
print("="*60)

bdc_ok = check_embedding_file(bdc_path)
psc_ok = check_embedding_file(psc_path)

print(f"\n{'='*60}")
if bdc_ok and psc_ok:
    print("✓ Both files are OK")
else:
    print("✗ One or both files are corrupted")
    print("\nAction needed: Re-run embedding generation notebook")

EMBEDDING FILE DIAGNOSTICS

Checking: /content/embeddings/bdc_embeddings_laberta_vulg.npy
  File size: 57.03 MB
  ✓ Loaded successfully
  Shape: (19466, 768)
  Total elements: 14,949,888

Checking: /content/embeddings/psc_embeddings_laberta_vulg.npy
  File size: 135.96 MB
  ✓ Loaded successfully
  Shape: (46408, 768)
  Total elements: 35,641,344

✓ Both files are OK
